In [ ]:
"""
# Feature Importance Analysis

## Load model and data
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

# Load model
model = joblib.load('../models/lightgbm_model.pkl')

# Load feature names
feature_names = joblib.load('../models/feature_names.pkl')

print(f"Model loaded. Features: {len(feature_names)}")

"""
## Feature Importance from Model
"""

# Get built-in feature importance
importance = model.feature_importances_

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'][:20], importance_df['importance'][:20])
plt.xlabel('Importance')
plt.title('Top 20 Features by Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

"""
## SHAP Analysis
"""

# Load sample data
X_sample = joblib.load('../data/sample_features.pkl')[:1000]

# Create SHAP explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.tight_layout()
plt.show()

"""
## Feature Interaction
"""

# Check feature interactions
interaction_values = shap.TreeExplainer(model).shap_interaction_values(X_sample)

# Plot interaction for top 2 features
top_features = importance_df['feature'][:2].tolist()
idx1 = feature_names.index(top_features[0])
idx2 = feature_names.index(top_features[1])

plt.figure(figsize=(10, 6))
shap.dependence_plot(
    idx1, shap_values, X_sample, 
    interaction_index=idx2,
    feature_names=feature_names,
    show=False
)
plt.title(f'Interaction: {top_features[0]} vs {top_features[1]}')
plt.show()